In [ ]:
#!pip install chromadb pypdf sentence-transformers langchain-community

# Day 11 — RAG Pipeline: Ask Your Documents
Building a Retrieval Augmented Generation system that answers questions from ML lecture notes.
Pipeline: Load PDF → Chunk → Embed → Store → Retrieve → Generate answer

In [3]:
import chromadb
import sentence_transformers
print("chromdb version", chromadb.__version__)
print("sentence Transformers version", sentence_transformers.__version__)

chromdb version 1.5.9
sentence Transformers version 5.5.1


In [18]:
from dotenv import load_dotenv
import os, pandas as pd
load_dotenv()
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

In [19]:
import os
os.environ['GROQ_API_KEY']="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"
key=os.getenv("GROQ_API_KEY")

In [22]:
llm=ChatGroq(api_key=os.getenv("GROQ_API_KEY"),
             model_name="llama-3.1-8b-instant",
             temperature=0.7
            )
             

In [27]:
loader=PyPDFLoader("Docs\Attention_paper.pdf")
pages=loader.load()
print("-"*50)
print(f"\n No. of Page: {len(pages)}")
print("-"*50)
print(pages[0].page_content[:500])
print("-"*50)
print(f"MetaData: {pages[0].metadata}")

--------------------------------------------------

 No. of Page: 15
--------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 
--------------------------------------------------
MetaData: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', '

## Step 2 — Split Text into Chunks
LLMs have token limits — we cannot pass 15 pages at once. Chunking splits text into small searchable pieces. chunk_overlap ensures no information is lost at boundaries.

In [42]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len)
chunks=splitter.split_documents(pages)
print(f" Total chunks : {len(chunks)}")
print(chunks[5].page_content)


 Total chunks : 93
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.


## Step 3 — Create Embeddings
Each chunk is converted to a 384-dimensional vector using HuggingFace's MiniLM model. Similar text = similar vectors. No API key needed — runs locally.

In [44]:
# Load embedding model — free, runs locally, no API key
print("Loading embedding model — first run downloads ~90MB...")
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print("Embedding model loaded ✓")
test_embed=embeddings.embed_query("what is neural network")
print(f"embedding dimension : {len(test_embed)}")
print(f"First 5 values: {test_embed[:5]}")

Loading embedding model — first run downloads ~90MB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded ✓
embedding dimension : 384
First 5 values: [-0.1085527166724205, -0.06544336676597595, 0.006954994518309832, 0.01917908899486065, -0.0013469342375174165]


In [49]:
# Create ChromaDB vector store from chunks
print("Creating ChromaDB vector store...")
vectorstore=Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="chromadb")
print(f"✅ Vector store created")
print(f" Total vector stored: {vectorstore._collection.count()}")
print("\n ============= Test Similarity Search=========")
test_query="what is attention you need"
results=vectorstore.similarity_search(test_query, k=3)
print(f"Query: {test_query}")
print(f"Top 3 retrieved chunks:\n")
for i, doc in enumerate(results, 1):
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')}):")
    print(doc.page_content[:200])
    print("-"*40)


Creating ChromaDB vector store...
✅ Vector store created
 Total vector stored: 465

 ============= Test Similarity Search=========
Query: what is attention you need
Top 3 retrieved chunks:

Chunk 1 (page 2):
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
3.2 Attent
----------------------------------------
Chunk 2 (page 2):
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
3.2 Attent
----------------------------------------
Chunk 3 (page 2):
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
3.2 Attent
----------------------------------------


## Step 4 — Build Full RAG Chain
Connecting retriever → prompt → LLM into one chain. The LLM can only answer from retrieved context — no hallucination.

In [50]:
# Step 1 — Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # fetch top 3 chunks
)

# Step 2 — RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an ML tutor assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say:
"I cannot find this in the provided lecture notes."
Do NOT make up information.

Context:
{context}"""),
    ("human", "{question}")
])

# Step 3 — Helper to format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Step 4 — Build complete RAG chain
rag_chain = (
    {"context" : retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready")
print("Flow: question → retriever → prompt → LLM → answer")

✅ RAG chain ready
Flow: question → retriever → prompt → LLM → answer


In [51]:
# Update these questions based on YOUR document content
questions = [
    "What is a neural network?",
    "How does backpropagation work?",
    "What is the difference between overfitting and underfitting?",
    "What activation functions are discussed in the notes?",
    "What is gradient descent and how is it used?"
]

# Out of scope question — tests hallucination prevention
out_of_scope = "What is the recipe for biryani?"

all_questions = questions + [out_of_scope]
results_log = []

for i, q in enumerate(all_questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {q}")
    print(f"{'='*60}")

    # Get retrieved chunks
    retrieved = retriever.invoke(q)

    # Get answer
    answer = rag_chain.invoke(q)
    print(f"📄 Retrieved {len(retrieved)} chunks")
    print(f"🤖 Answer: {answer}")

    results_log.append({
        "question"        : q,
        "answer"          : answer,
        "chunks_retrieved": len(retrieved),
        "in_scope"        : i <= len(questions)
    })

print("\n✅ All questions answered")


Q1: What is a neural network?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

Q2: How does backpropagation work?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

Q3: What is the difference between overfitting and underfitting?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

Q4: What activation functions are discussed in the notes?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

Q5: What is gradient descent and how is it used?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

Q6: What is the recipe for biryani?
📄 Retrieved 3 chunks
🤖 Answer: I cannot find this in the provided lecture notes.

✅ All questions answered


In [52]:
os.makedirs('data', exist_ok=True)

# Save Q&A results
df_results = pd.DataFrame(results_log)
df_results.to_csv('data/day11_rag_results.csv', index=False)
print("Results saved ✓")

# Summary
print("\n=== Day 11 Summary ===")
print(f"PDF pages loaded   : {len(pages)}")
print(f"Chunks created     : {len(chunks)}")
print(f"Vectors in ChromaDB: {vectorstore._collection.count()}")
print(f"Questions answered : {len(questions)}")
print(f"Hallucination test : ✅ Passed")
print("\n🚀 RAG pipeline complete — ready for Day 12 Streamlit app!")

Results saved ✓

=== Day 11 Summary ===
PDF pages loaded   : 15
Chunks created     : 93
Vectors in ChromaDB: 465
Questions answered : 5
Hallucination test : ✅ Passed

🚀 RAG pipeline complete — ready for Day 12 Streamlit app!
